# 02 Feature Transformation & Target Generation


 Responsibility
 --------------
 Base Feature Dataset을 입력으로 받아:

   1. Generic Validation
   2. Feature Transformation
   3. Learning 데이터셋의 RUL Target Generation
   4. Feature / Target Candidate 분리 및 결과 확인
   5. Engineered Dataset 저장
```
 Learning Dataset
   Base Feature
       ↓
   Generic Validation
       ↓
   Feature Transformation
       ↓
   RUL Target Generation
       ↓
   Engineered Learning Dataset

 Test Dataset
   Base Feature
       ↓
   Generic Validation
       ↓
   Feature Transformation
       ↓
   Engineered Test Dataset
```
 이 Notebook은 다음을 수행하지 않는다.
   - EDA
   - Feature Ranking
   - Feature Selection
   - Target Selection
   - Scaling
   - Sequence / Windowing
   - Dataset Builder

In [ ]:
# =============================================================================
# 0. Imports
# =============================================================================
import os
import warnings

import pandas as pd

from utils.project_utils import *
from utils.bearing_utils import (
    META_COLUMNS, TARGET_CANDIDATES, generate_rul_targets
)
from utils.feature_utils import RDI_FEATURES, build_damage_features

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


# =============================================================================
# 1. Path Configuration
# =============================================================================
LEARNING_INPUT_PATH = "learning_set_features.csv"
TEST_INPUT_PATH = "test_set_features.csv"

OUTPUT_PATH = "outputs/02_feature_transformation/"
LEARNING_OUTPUT_PATH = OUTPUT_PATH + "learning_engineered_features.csv"
TEST_OUTPUT_PATH = OUTPUT_PATH + "test_engineered_features.csv"


# =============================================================================
# 2. Configuration
# =============================================================================
# Feature Transformation 정책.
#
# RDI_FEATURES는 Feature Selection 결과가 아니라
# Transformation을 적용할 원본 Feature의 기본 목록이다.
TRANSFORM_FEATURES = RDI_FEATURES

BASELINE_RATIO = 0.05
THRESHOLD_K = 0.0

In [ ]:
# =============================================================================
# 3. Dataset Loading
# =============================================================================
def load_dataset(path: str) -> pd.DataFrame:
    """
    Load a Base Feature Dataset.

    Parameters
    ----------
    path : str
        CSV path.

    Returns
    -------
    pd.DataFrame
        Loaded dataset.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset file not found: {path}")

    df = pd.read_csv(path)
    print(
        f"Loaded: {path} "
        f"({df.shape[0]:,} rows × {df.shape[1]} columns)"
    )

    return df

learning_df = load_dataset(LEARNING_INPUT_PATH)
test_df = load_dataset(TEST_INPUT_PATH)

In [ ]:
# =============================================================================
# 4. Generic Validation
# =============================================================================
def validate_base_dataset(
    df: pd.DataFrame, dataset_name: str
) -> pd.DataFrame:
    """
    Perform generic schema and temporal validation.
    Domain-specific target validation is intentionally excluded.

    Parameters
    ----------
    df : pd.DataFrame
        Base Feature Dataset.
    dataset_name : str
        Dataset label used for logging.

    Returns
    -------
    pd.DataFrame
        Time-sorted validated dataset.
    """
    print("=" * 60)
    print(f"[{dataset_name}] Generic Validation")
    print("=" * 60)

    # -------------------------------------------------------------------------
    # Required Metadata
    # -------------------------------------------------------------------------
    assert_required_columns(df, META_COLUMNS)

    # -------------------------------------------------------------------------
    # Time Ordering
    # -------------------------------------------------------------------------
    df = sort_by_time(df, bearing_col="bearing", time_col="elapsed_sec")

    assert_monotonic_order(
        df, group_col="bearing", order_col="elapsed_sec"
    )

    # -------------------------------------------------------------------------
    # Basic Data Integrity
    # -------------------------------------------------------------------------
    duplicate_count = df.duplicated().sum()
    numeric_columns = df.select_dtypes(include="number").columns

    inf_summary = {
        column: int(df[column].isin([float("inf"), float("-inf")]).sum())
        for column in numeric_columns
    }

    inf_summary = {
        column: count
        for column, count in inf_summary.items()
        if count > 0
    }

    if duplicate_count:
        print(f"[Warning] Duplicate rows: {duplicate_count:,}")

    if inf_summary:
        print("[Warning] Infinite values detected:")
        for column, count in inf_summary.items():
            print(f"  - {column}: {count:,}")
    else:
        print("Infinite values: none")

    print(
        f"Rows    : {len(df):,}\n"
        f"Columns : {len(df.columns):,}"
    )

    print("Generic validation passed.")
    return df

learning_df = validate_base_dataset(learning_df, "Learning")
test_df = validate_base_dataset(test_df, "Test")

In [ ]:
# =============================================================================
# 5. Feature / Metadata Overview
# =============================================================================
def show_column_overview(df: pd.DataFrame, dataset_name: str) -> None:
    """
    Display the semantic column classification available at 02 stage.
    Target Candidates are not expected to exist before target generation.
    """
    _, feature_columns = split_columns(df, META_COLUMNS)

    print("=" * 60)
    print(f"[{dataset_name}] Base Feature Overview")
    print("=" * 60)

    print(f"Metadata columns : {len(META_COLUMNS)}")
    print(f"Feature columns  : {len(feature_columns)}")

    print("\nTransformation targets:")
    for feature in TRANSFORM_FEATURES:
        status = "available" if feature in df.columns else "missing"
        print(f"  - {feature}: {status}")

show_column_overview(learning_df, "Learning")
show_column_overview(test_df, "Test")

In [ ]:
# =============================================================================
# 6. Feature Transformation
# =============================================================================
#
# Learning과 Test에 동일한 Transformation 정책을 독립적으로 적용한다.
#
# 중요한 계약:
#   - dataset_type 인자를 사용하지 않는다.
#   - Learning에서 계산한 baseline/threshold parameter를 Test에 전달하지 않는다.
#   - 공유되는 것은 Transformation 정책뿐이다.
# =============================================================================
print("=" * 60)
print("Feature Transformation")
print("=" * 60)

learning_df = build_damage_features(
    learning_df,
    features=TRANSFORM_FEATURES,
    baseline_ratio=BASELINE_RATIO,
    threshold_k=THRESHOLD_K,
)

print(
    "Learning transformation complete: "
    f"{learning_df.shape[0]:,} rows × "
    f"{learning_df.shape[1]:,} columns"
)

test_df = build_damage_features(
    test_df,
    features=TRANSFORM_FEATURES,
    baseline_ratio=BASELINE_RATIO,
    threshold_k=THRESHOLD_K,
)

print(
    "Test transformation complete: "
    f"{test_df.shape[0]:,} rows × "
    f"{test_df.shape[1]:,} columns"
)

In [ ]:
# =============================================================================
# 7. RUL Target Generation — Learning Only
# =============================================================================
#
# Target Candidates:
#   life_ratio
#   RUL_sec
#   RUL_norm
#
# Test에는 Target Generation을 수행하지 않는다.
# =============================================================================
print("=" * 60)
print("RUL Target Generation — Learning")
print("=" * 60)

learning_df = generate_rul_targets(
    learning_df,
    bearing_col="bearing",
    time_col="elapsed_sec",
)

print("RUL Target Candidates generated:")
print(
    learning_df[
        ["bearing", "elapsed_sec", "failure_time", *TARGET_CANDIDATES]
    ].head(5)
)

In [ ]:
# =============================================================================
# 8. Feature / Target Candidate Separation
# =============================================================================
def classify_columns(
    df: pd.DataFrame, include_targets: bool
) -> tuple[list[str], list[str], list[str]]:
    """
    Classify columns into Metadata, Target Candidates, and Feature Candidates.

    Parameters
    ----------
    df : pd.DataFrame
        Engineered dataset.
    include_targets : bool
        Whether Target Candidates are present in the dataset.

    Returns
    -------
    tuple[list[str], list[str], list[str]]
        metadata_columns,
        target_columns,
        feature_columns
    """
    metadata_columns = [
        column for column in META_COLUMNS
        if column in df.columns
    ]

    target_columns = []
    if include_targets:
        target_columns = [
            column for column in TARGET_CANDIDATES
            if column in df.columns
        ]

    excluded_columns = (
        metadata_columns + target_columns +
        ["failure_time"] if "failure_time" in df.columns else []
    )

    feature_columns = [
        column for column in df.columns
        if column not in excluded_columns
    ]

    return metadata_columns, target_columns, feature_columns

learning_meta, learning_targets, learning_features = (
    classify_columns(learning_df, include_targets=True)
)

test_meta, test_targets, test_features = (
    classify_columns(test_df, include_targets=False)
)

In [ ]:
# =============================================================================
# 9. Candidate Overview
# =============================================================================
print("=" * 60)
print("Engineered Dataset Candidate Overview")
print("=" * 60)

print("\n[Learning]")
print(f"Metadata candidates : {len(learning_meta)}")
print(f"Target candidates   : {len(learning_targets)}")
print(f"Feature candidates  : {len(learning_features)}")

print("\nTarget Candidates:")
for column in learning_targets:
    print(f"  - {column}")

print("\nDerived Feature Candidates:")
derived_learning_features = [
    column for column in learning_features
    if column.startswith("Rectified_")
    or column.startswith("RDI_")
]

for column in derived_learning_features:
    print(f"  - {column}")

print("\n[Test]")
print(f"Metadata candidates : {len(test_meta)}")
print(f"Target candidates   : {len(test_targets)}")
print(f"Feature candidates  : {len(test_features)}")

print("\nDerived Feature Candidates:")
derived_test_features = [
    column for column in test_features
    if column.startswith("Rectified_")
    or column.startswith("RDI_")
]

for column in derived_test_features:
    print(f"  - {column}")

In [ ]:
# =============================================================================
# 10. Target Generation Integrity Check
# =============================================================================
assert_required_columns(learning_df, TARGET_CANDIDATES)

if test_targets:
    raise ValueError(
        "Test dataset must not contain Target Candidates "
        "at the end of 02."
    )

print("=" * 60)
print("Target Generation Validation")
print("=" * 60)
print("Learning: Target Candidates present.")
print("Test    : Target Candidates absent.")

In [ ]:
# =============================================================================
# 11. Final Output Schema Check
# =============================================================================
def validate_engineered_output(
    df: pd.DataFrame, dataset_name: str, expect_targets: bool
) -> None:
    """
    Validate the final Engineered Dataset contract.
    """
    assert_required_columns(df, META_COLUMNS)

    if expect_targets:
        assert_required_columns(df, TARGET_CANDIDATES)

    if not expect_targets:
        unexpected_targets = [
            column for column in TARGET_CANDIDATES
            if column in df.columns
        ]

        if unexpected_targets:
            raise ValueError(
                f"{dataset_name} contains unexpected "
                f"Target Candidates: {unexpected_targets}"
            )

    print(f"{dataset_name} output schema validation passed.")

validate_engineered_output(
    learning_df, "Learning", expect_targets=True
)

validate_engineered_output(
    test_df, "Test", expect_targets=False
)

In [ ]:
# =============================================================================
# 12. Save Engineered Datasets
# =============================================================================
os.makedirs(OUTPUT_PATH, exist_ok=True)
learning_df.to_csv(LEARNING_OUTPUT_PATH, index=False)
test_df.to_csv(TEST_OUTPUT_PATH, index=False)

print("=" * 60)
print("02 Pipeline Complete")
print("=" * 60)

print(f"Learning output : {LEARNING_OUTPUT_PATH}")
print(f"Test output     : {TEST_OUTPUT_PATH}")

print(
    "Learning shape  : "
    f"{learning_df.shape[0]:,} × {learning_df.shape[1]:,}"
)

print(
    "Test shape      : "
    f"{test_df.shape[0]:,} × {test_df.shape[1]:,}"
)